# Manuscript 1959239 — 査読者3への対応のための追加解析ノートブック

査読者3の指摘のうち、**新しい計測が必要な3点**を確かめます。学習済みcheckpointは、これまでと同じもの(`logs/rsl_rl/g1_velocity/*_seed{1..8}`)を使います。

| 節 | 内容 | 査読者3の指摘 |
|---|---|---|
| §R1 | 転倒判定に使っている体の部位(根元の剛体=骨盤か)と、判定式を環境の設定から直接確認 | Major 2 |
| §R2 | 減衰なし(α = 1)で歩かせ、右側と左側の「追従誤差・トルク・仕事率・指令の振れ幅」を比較。あわせて実際の歩行速度を計測 | Major 4、Minor 5 |
| §R3 | §R2の左右差(seedごと)が、seedごとの転倒閾値の左右差と対応しているかを確認 | Major 4 |
| §R4 | 結果の保存(revision_outputs/R3_*.csv, R3_*.json) | — |

実行の目安:T4 GPU で 30〜45 分(8 seed × 4 速度 × 10 初期条件)。途中で止まっても、続きから再開できます。

## 1. セットアップ

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = ""
os.environ["WANDB_MODE"] = "offline"

%pip install -q "mjlab==1.2.0" "mujoco==3.5.0" "warp-lang==1.12.1" "numpy>=2.0,<2.3" scipy mediapy pandas

import torch
assert torch.cuda.is_available(), "GPU ランタイムに切り替えてください"
print("✅ install done:", torch.cuda.get_device_name(0))

### 1-2. 学習済みcheckpointと元の結果CSVを、Googleドライブから復元する

新しいランタイムには `logs/` フォルダ(学習済みcheckpoint)がありません。学習ノートブックが退避した
`MyDrive/g1_cross_seed_logs/logs` を、作業フォルダ(`/content`)にコピーします。
あわせて、元の実験の結果CSV(`g1_hemiparesis_fine.csv` など)を、ドライブ内のよく使う場所(マイドライブ直下・g1_cross_seed_logs・Colab Notebooks など)から探して、同じ場所にコピーします。

**実行すると、Googleアカウントへのアクセス許可を求める画面が出ます。学習に使ったのと同じアカウントを選んで「許可」してください。**

In [ ]:
import os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content")

DRIVE_LOGS = "/content/drive/MyDrive/g1_cross_seed_logs/logs"   # 学習ノートブックの退避先
if not os.path.isdir("logs/rsl_rl/g1_velocity"):
    if os.path.isdir(DRIVE_LOGS):
        print("ドライブから logs をコピーしています(数分かかることがあります)...")
        shutil.copytree(DRIVE_LOGS, "/content/logs", dirs_exist_ok=True)
    else:
        print(f"⚠️ {DRIVE_LOGS} が見つかりません。ドライブ内の場所を確認し、DRIVE_LOGS を書き換えてください。")

found = sorted(glob.glob("logs/rsl_rl/g1_velocity/*_seed*"))
print(f"見つかった実行フォルダ: {len(found)} 個")
for s in range(1, 9):
    print(f"  seed{s}: {'OK' if any(p.endswith(f'_seed{s}') for p in found) else '見つかりません'}")

# 元の結果CSVを、ドライブ内のよく使う場所から探してコピーする(見つからなくても、この後の解析は動きます)
for name in ["g1_hemiparesis_fine.csv", "g1_hemiparesis_speed_dependence.csv", "g1_hemiparesis_coarse.csv"]:
    if os.path.exists(name):
        print(f"{name}: 既にあります"); continue
    # ドライブ全体を再帰的に探すと非常に遅いため、よく使う場所だけを探す
    cands = ["/content/drive/MyDrive", "/content/drive/MyDrive/g1_cross_seed_logs",
             "/content/drive/MyDrive/g1_hemiparesis", "/content/drive/MyDrive/Colab Notebooks"]
    hits = [os.path.join(c, name) for c in cands if os.path.exists(os.path.join(c, name))]
    if hits:
        shutil.copy(hits[0], name); print(f"{name}: {hits[0]} からコピーしました")
    else:
        print(f"{name}: ドライブ内に見つかりません(手動でアップロードしてください)")

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("mjlab") is None:
    print("mjlab が見つかりません。インストールします(2〜3分)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "mjlab==1.2.0", "mujoco==3.5.0",
                        "warp-lang==1.12.1", "numpy>=2.0,<2.3", "scipy", "mediapy", "pandas"],
                       capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-1500:])
    importlib.invalidate_caches()
    assert importlib.util.find_spec("mjlab") is not None, "インストールに失敗しました。上の表示を確認してください。"
    print("✅ mjlab をインストールしました")

import math, glob, re, json, time
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import asdict
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
ENV_SEED = 100
RIGHT_FACING_YAW_RAD = -math.pi / 2
FALL_LIMIT_RAD = math.radians(70.0)   # mjlab velocity task の fell_over と同じ閾値
SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
STEPS = 250
OUT_DIR = Path("revision_outputs"); OUT_DIR.mkdir(exist_ok=True)

def latest_ckpt(seed):
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    assert run_dirs, f"seed{seed}: runディレクトリが見つかりません"
    return max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

_probe_cfg = load_env_cfg(TASK, play=True)
_probe_cfg.scene.num_envs = 1
_probe_env = ManagerBasedRlEnv(cfg=_probe_cfg, device=DEVICE, render_mode=None)
JOINT_NAMES = list(_probe_env.scene["robot"].joint_names)
LEFT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("left_")]
RIGHT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("right_")]
assert len(JOINT_NAMES) == 29 and len(LEFT_HALF_JOINTS) == 13 and len(RIGHT_HALF_JOINTS) == 13
HEMI_CONDITIONS = {
    "right_half_paresis": [JOINT_NAMES.index(n) for n in RIGHT_HALF_JOINTS],
    "left_half_paresis":  [JOINT_NAMES.index(n) for n in LEFT_HALF_JOINTS],
}
_probe_env.close(); del _probe_env; torch.cuda.empty_cache()
print("中央(体幹)の関節:", [n for n in JOINT_NAMES if not n.startswith(("left_", "right_"))])

In [ ]:
def make_env_and_policy(checkpoint_path, joint_idx_list, level, env_seed=ENV_SEED, num_envs=1,
                        remove_fall_termination=False, obs_noise=False):
    # 元のノートブックと同じ構成。remove_fall_termination=True のときだけ fell_over 終了条件を外す。
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = num_envs
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    if remove_fall_termination:
        play_cfg.terminations.pop("fell_over", None)
    if obs_noise:
        play_cfg.observations["actor"].enable_corruption = True
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)
    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)
    with torch.no_grad():
        final_layer = runner.alg.actor.mlp[-1]
        for idx in joint_idx_list:
            final_layer.weight[idx, :] *= level
            final_layer.bias[idx] *= level
    return runner, policy, env, raw_env, agent_cfg

def close_all(*objs):
    for o in objs:
        try:
            o.close()
        except Exception:
            pass
    torch.cuda.empty_cache()

def tilt_rad(raw_env):
    pg = raw_env.scene["robot"].data.projected_gravity_b
    return torch.acos((-pg[:, 2]).clamp(-1.0, 1.0)).abs()

def joint_targets(raw_env):
    term = raw_env.action_manager.get_term("joint_pos")
    return term.raw_action * term.scale + term.offset

## §R1. 転倒判定の対象と判定式の確認(Major 2)

mjlab の `fell_over` 終了条件の中身(関数と引数)と、ロボットの根元の剛体(root body)の名前を表示します。
このノートブックと改訂版の解析では、`projected_gravity_b`(根元の剛体の座標系で見た重力方向)から傾きを計算しています。

In [ ]:
import inspect
_cfg = load_env_cfg(TASK, play=True)
ft = _cfg.terminations["fell_over"]
print("fell_over の関数:", getattr(ft.func, "__module__", ""), getattr(ft.func, "__name__", ft.func))
print("fell_over の引数:", ft.params)
try:
    print(inspect.getsource(ft.func))
except Exception as e:
    print("ソース表示不可:", e)

runner, policy, env, raw_env, agent_cfg = make_env_and_policy(latest_ckpt(1), [], 1.0)
robot = raw_env.scene["robot"]
m = raw_env.sim.mj_model
import mujoco
root_body_id = int(m.jnt_bodyid[0])          # 浮遊ベース(free joint)が付いている剛体
R1_INFO = {
    "fell_over_func": f"{getattr(ft.func, '__module__', '')}.{getattr(ft.func, '__name__', '')}",
    "fell_over_params": {k: (float(v) if isinstance(v, (int, float)) else str(v)) for k, v in ft.params.items()},
    "free_joint_type_is_free": bool(m.jnt_type[0] == mujoco.mjtJoint.mjJNT_FREE),
    "root_body_name": mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_BODY, root_body_id),
    "robot_body_names_first5": list(robot.body_names[:5]),
    "limit_deg_used_here": math.degrees(FALL_LIMIT_RAD),
}
for k, v in R1_INFO.items():
    print(f"{k:26s}: {v}")
close_all(env); del runner, policy, env, raw_env
print("\n→ root_body_name が 'pelvis' で、fell_over が bad_orientation(limit_angle=70°)であれば、")
print("  転倒の定義は『骨盤(根元の剛体)の鉛直軸と重力方向のなす角が 70° を超えること』と確定します。")

## §R2. 減衰なしでの左右差と、実際の歩行速度(Major 4、Minor 5)

減衰なし(α = 1)で、4 つの指令速度(0.3, 0.5, 0.7, 0.9 m/s)× 8 seed × 10 初期条件(§D と同じ並列 10 環境)を歩かせます。
転倒による終了は外し(リセットが起きないように)、同じ判定式で転倒を検出します。各制御ステップで次を記録し、
**定常歩行区間(制御ステップ 50〜250)**で平均します。

- 追従誤差:|q* − q|(指令した目標角と実際の角度の差、度)
- トルク:|τ|(N·m)と、上限に対する割合 |τ|/τ_max
- 仕事率:|τ · q̇|(W)
- 指令の振れ幅:|q* − q⁰|(既定姿勢から指令がどれだけ離れているか、度)。減衰は、まさにこの量を α 倍にします
- 実際の前進速度:胴体の座標系での前向き速度(m/s)

トルクは、PD の式 τ = clip(K_p(q* − q) − K_d q̇, ±τ_max) で関節ごとに計算し、mjlab のアクチュエータ出力
(`actuator_force`)を**関節の順番に並べ替えたもの**と一致するか(相関)を確認します。
1 回目の実行ではトルクの左右差が追従誤差と食い違っており、アクチュエータ出力の並び順が関節の順番と違っていた可能性が高いため、v2 で並べ替えと照合を加えました。
左右の比較は、片側 13 関節(脚 6 + 腕 7)全体と、脚 6 関節だけ、の 2 通りで行います。

In [ ]:
import mujoco
SPEEDS = [0.3, 0.5, 0.7, 0.9]
N_IC = 10
STEADY = slice(50, 250)
R2_CSV = OUT_DIR / "R3_baseline_side_metrics_v2.csv"   # v2: トルクの関節対応を修正

LEFT_IDX = [JOINT_NAMES.index(n) for n in LEFT_HALF_JOINTS]
RIGHT_IDX = [JOINT_NAMES.index(n) for n in RIGHT_HALF_JOINTS]
LEG_SUFFIX = ("hip_pitch", "hip_roll", "hip_yaw", "knee", "ankle_pitch", "ankle_roll")
LEFT_LEG = [JOINT_NAMES.index(n) for n in LEFT_HALF_JOINTS if any(s in n for s in LEG_SUFFIX)]
RIGHT_LEG = [JOINT_NAMES.index(n) for n in RIGHT_HALF_JOINTS if any(s in n for s in LEG_SUFFIX)]
assert len(LEFT_LEG) == len(RIGHT_LEG) == 6
# 左右の対応(left_xxx ↔ right_xxx)が同じ順番になっていることを確認
assert [JOINT_NAMES[i][5:] for i in LEFT_IDX] == [JOINT_NAMES[i][6:] for i in RIGHT_IDX]

def actuator_joint_order(raw_env):
    # mjlab の actuator は関節の順番ではなく、モーターの種類ごとに並んでいることがある。
    # 各 actuator が動かす関節(actuator_trnid)から、関節順(JOINT_NAMES)への並べ替えを作る。
    m = raw_env.sim.mj_model
    perm = np.full(29, -1, dtype=int)
    for a in range(m.nu):
        jname = (mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_JOINT, int(m.actuator_trnid[a, 0])) or "").split("/")[-1]
        if jname in JOINT_NAMES:
            perm[JOINT_NAMES.index(jname)] = a
    assert (perm >= 0).all(), "actuator と関節の対応が取れません"
    return perm

def pd_arrays(raw_env):
    m = raw_env.sim.mj_model
    perm = actuator_joint_order(raw_env)
    KP = np.array([float(m.actuator_gainprm[a, 0]) for a in perm])
    KD = np.array([float(-m.actuator_biasprm[a, 2]) for a in perm])
    TM = np.array([float(m.actuator_forcerange[a, 1]) if m.actuator_forcelimited[a] else np.inf for a in perm])
    return KP, KD, TM

def actuator_torque(robot):
    for name in ["actuator_force", "applied_torque"]:
        if hasattr(robot.data, name):
            v = getattr(robot.data, name)
            if torch.is_tensor(v) and v.shape[-1] == 29:
                return v, name
    return None, None

def forward_speed(robot):
    # 胴体の座標系での前向き速度(指令と同じ座標系)。mjlabの版によって属性名が違う場合に備える
    for name in ["root_link_lin_vel_b", "root_lin_vel_b"]:
        if hasattr(robot.data, name):
            return getattr(robot.data, name)[:, 0].detach().cpu().numpy().copy()
    raise AttributeError("胴体速度の属性が見つかりません: " + ", ".join(a for a in dir(robot.data) if "vel" in a))

def baseline_rollout(ckpt, speed):
    runner, policy, env, raw_env, _ = make_env_and_policy(ckpt, [], 1.0, num_envs=N_IC,
                                                          remove_fall_termination=True)
    robot = raw_env.scene["robot"]
    KP, KD, TM = pd_arrays(raw_env)
    q0 = robot.data.default_joint_pos.detach().cpu().numpy().copy()
    cmd = torch.tensor([[speed, 0.0, 0.0]], device=DEVICE).repeat(N_IC, 1)
    obs = env.get_observations()
    fall_step = np.full(N_IC, -1, dtype=int)
    Q, QD, T, TAU, VX = [], [], [], [], []
    TAU_PD, TAU_RAW, TAU_RAW_ID = [], [], []
    PERM = actuator_joint_order(raw_env)
    tau_source = None
    for step in range(STEPS):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = cmd
        with torch.no_grad():
            obs, _, _, _ = env.step(policy(obs))
        tilt = tilt_rad(raw_env).detach().cpu().numpy()
        newly = (fall_step < 0) & (tilt > FALL_LIMIT_RAD)
        fall_step[newly] = step + 1
        q = robot.data.joint_pos.detach().cpu().numpy().copy()
        qd = robot.data.joint_vel.detach().cpu().numpy().copy()
        tgt = joint_targets(raw_env).detach().cpu().numpy().copy()
        tau_pd = np.clip(KP * (tgt - q) - KD * qd, -TM, TM)        # 関節順。制御ステップ時点のPDの式
        tau_t, src = actuator_torque(robot)
        if tau_t is not None:
            raw = tau_t.detach().cpu().numpy().copy()
            TAU_RAW.append(raw[:, PERM]); TAU_RAW_ID.append(raw)
        TAU_PD.append(tau_pd)
        tau = tau_pd
        Q.append(q); QD.append(qd); T.append(tgt); TAU.append(tau)
        VX.append(forward_speed(robot))
    close_all(env); del runner, policy, env, raw_env
    # どのトルクを使うかを、PDの式との一致度で決める(関節の並びの取り違えを防ぐ)
    tau_pd_all = np.stack(TAU_PD, 1)
    tau_use, tau_source, agree = tau_pd_all, "pd_formula", {}
    if TAU_RAW:
        mapped, ident = np.stack(TAU_RAW, 1), np.stack(TAU_RAW_ID, 1)
        corr = lambda a, b: float(np.corrcoef(a[:, STEADY].ravel(), b[:, STEADY].ravel())[0, 1])
        agree = {"actuator_force_mapped_vs_pd": corr(mapped, tau_pd_all),
                 "actuator_force_unmapped_vs_pd": corr(ident, tau_pd_all)}
        if agree["actuator_force_mapped_vs_pd"] > 0.9:
            tau_use, tau_source = mapped, "actuator_force_mapped_to_joints"
    return dict(q=np.stack(Q, 1), qd=np.stack(QD, 1), tgt=np.stack(T, 1), tau=tau_use,
                vx=np.stack(VX, 1), q0=q0, fall_step=fall_step, TM=TM, tau_source=tau_source, agree=agree)

def side_metrics(tr, idx):
    s = STEADY
    err = np.degrees(np.abs(tr["tgt"][:, s][..., idx] - tr["q"][:, s][..., idx]))
    amp = np.degrees(np.abs(tr["tgt"][:, s][..., idx] - tr["q0"][:, None, idx]))
    tau = np.abs(tr["tau"][:, s][..., idx])
    tau_rel = tau / tr["TM"][idx]
    pw = np.abs(tr["tau"][:, s][..., idx] * tr["qd"][:, s][..., idx])
    # 各初期条件(行)ごとに、時間と関節で平均 → 初期条件 10 個の平均
    f = lambda a: float(a.mean(axis=(1, 2)).mean())
    return {"track_err_deg": f(err), "cmd_amp_deg": f(amp), "torque_Nm": f(tau),
            "torque_rel": f(tau_rel), "power_W": f(pw)}

done = set()
if R2_CSV.exists():
    _d = pd.read_csv(R2_CSV); done = set(zip(_d["seed"], _d["speed"]))

TAU_SOURCE = None
for seed in SEEDS:
    ckpt = latest_ckpt(seed)
    for speed in SPEEDS:
        if (seed, speed) in done:
            continue
        t0 = time.time()
        tr = baseline_rollout(ckpt, speed)
        TAU_SOURCE = tr["tau_source"]
        rows = []
        for group, (li, ri) in {"side13": (LEFT_IDX, RIGHT_IDX), "leg6": (LEFT_LEG, RIGHT_LEG)}.items():
            L, R = side_metrics(tr, li), side_metrics(tr, ri)
            for k in L:
                rows.append({"seed": seed, "speed": speed, "group": group, "metric": k,
                             "left": L[k], "right": R[k]})
        vx = tr["vx"][:, STEADY].mean(axis=1)
        rows.append({"seed": seed, "speed": speed, "group": "whole", "metric": "measured_speed_mps",
                     "left": float(vx.mean()), "right": float(vx.std())})   # left=平均, right=初期条件間SD
        rows.append({"seed": seed, "speed": speed, "group": "whole", "metric": "falls_out_of_10",
                     "left": int((tr["fall_step"] > 0).sum()), "right": np.nan})
        for k, v in tr["agree"].items():
            rows.append({"seed": seed, "speed": speed, "group": "whole", "metric": "corr_" + k, "left": v, "right": np.nan})
        rows.append({"seed": seed, "speed": speed, "group": "whole", "metric": f"tau_source={TAU_SOURCE}",
                     "left": np.nan, "right": np.nan})
        pd.DataFrame(rows).to_csv(R2_CSV, mode="a", header=not R2_CSV.exists(), index=False)
        print("  トルクの一致度(相関):", {k: round(v, 3) for k, v in tr["agree"].items()})
        print(f"seed{seed} {speed} m/s: falls {int((tr['fall_step'] > 0).sum())}/10, "
              f"実速度 {vx.mean():.3f} m/s, torque={TAU_SOURCE} ({time.time()-t0:.0f}s)")
print("完了")

In [ ]:
from scipy import stats
d = pd.read_csv(R2_CSV)
d = d[~d["metric"].str.startswith(("tau_source", "corr_"))]

# --- 実際の歩行速度(Minor 5) ---
sp = d[d["metric"] == "measured_speed_mps"].pivot_table(index="seed", columns="speed", values="left")
print("定常区間(制御ステップ50〜250)の実際の前進速度 [m/s](行=seed、列=指令速度)")
print(sp.round(3).to_string())
print("\n指令速度ごとの平均 ± SD(seed間):")
print(pd.DataFrame({"mean": sp.mean(), "sd": sp.std(), "min": sp.min(), "max": sp.max()}).round(3).to_string())
falls = d[d["metric"] == "falls_out_of_10"]["left"].sum()
print(f"\n減衰なしでの転倒:合計 {int(falls)} / {len(SEEDS) * len(SPEEDS) * N_IC} 試行")

# --- 左右差(Major 4) ---
side = d[d["group"].isin(["side13", "leg6"])].copy()
side["AI"] = (side["right"] - side["left"]) / ((side["right"] + side["left"]) / 2) * 100   # 非対称指数(%)、正=右が大きい
rows = []
for (group, metric, speed), g in side.groupby(["group", "metric", "speed"]):
    diff = g["right"] - g["left"]
    n_r = int((diff > 0).sum()); n_l = int((diff < 0).sum())
    try:
        p_w = stats.wilcoxon(g["right"], g["left"], method="exact").pvalue
    except Exception:
        p_w = np.nan
    rows.append({"group": group, "metric": metric, "speed": speed,
                 "left_mean": g["left"].mean(), "right_mean": g["right"].mean(),
                 "AI_mean_%": g["AI"].mean(), "AI_min_%": g["AI"].min(), "AI_max_%": g["AI"].max(),
                 "seeds_R>L": n_r, "seeds_L>R": n_l, "wilcoxon_p": p_w})
R2_SUMMARY = pd.DataFrame(rows)
pd.set_option("display.width", 200)
print("\n減衰なしでの左右比較(8 seed、seedごとに10初期条件の平均):")
print(R2_SUMMARY.round(3).to_string(index=False))
R2_SUMMARY.to_csv(OUT_DIR / "R3_baseline_side_summary_v2.csv", index=False)

## §R3. seedごとの左右差は、seedごとの転倒閾値の左右差と対応するか(Major 4)

seedごとの転倒閾値の左右差(右 − 左)を、2 通りで求めます。

- 反復試行(§D、10 初期条件)から求めた 50% 閾値の差
- 1 回ロールアウトの細かいスイープ(表2)から求めた、方策ごとの閾値の差

これを、§R2 の減衰なしでの非対称指数(0.5 m/s)と Spearman の順位相関で比べます。
8 seed しかないため、この相関は「強い対応があるかどうか」の目安にとどまります。

In [ ]:
# 反復試行の結果:手元(revision_outputs)になければ GitHub の公開データを使う
VAR_CSV = OUT_DIR / "D_variability.csv"
if not VAR_CSV.exists():
    url = "https://raw.githubusercontent.com/rehabiliworld1-max/fall-threshold-unilateral-attenuation/main/results/D_variability.csv"
    pd.read_csv(url).to_csv(VAR_CSV, index=False); print("GitHub から取得:", url)
var = pd.read_csv(VAR_CSV)
var["fell_over"] = var["fell_over"].astype(str).str.lower() == "true"

def thr50(sub):
    p = sub.groupby("level")["fell_over"].mean().sort_index(ascending=False)   # α 高 → 低
    lv, pv = p.index.values, p.values
    if pv[0] >= 0.5: return float(lv[0])
    for i in range(1, len(lv)):
        if pv[i] >= 0.5:
            return float(lv[i-1] + (0.5 - pv[i-1]) * (lv[i] - lv[i-1]) / (pv[i] - pv[i-1]))
    return float(lv[-1])

thr = pd.DataFrame({c: {sd: thr50(g) for sd, g in var[var["condition"] == c].groupby("seed")}
                    for c in ["right_half_paresis", "left_half_paresis"]})
thr["diff_R_minus_L_pp"] = (thr["right_half_paresis"] - thr["left_half_paresis"]) * 100

# 1回ロールアウト(表2)の方策ごとの閾値:転倒しなかった最も低い α(改訂版§Cの結果ファイル)
single = None
for cand in [OUT_DIR / "C_mechanism_summary.csv",
             "https://raw.githubusercontent.com/rehabiliworld1-max/fall-threshold-unilateral-attenuation/main/results/C_mechanism_summary.csv"]:
    try:
        single = pd.read_csv(cand); break
    except Exception:
        pass
if single is not None:
    single = single[single["level"] < 1.0].copy()
    single["fell_over"] = single["fell_over"].astype(str).str.lower() == "true"
    st = single[~single["fell_over"]].groupby(["condition", "seed"])["level"].min().unstack(0)
    thr["single_diff_R_minus_L_pp"] = (st["right_half_paresis"] - st["left_half_paresis"]) * 100
else:
    print("1回ロールアウトの結果ファイルが見つからないため、反復試行の閾値だけで比べます。")

base = side[np.isclose(side["speed"], 0.5)]
rows = []
for (group, metric), g in base.groupby(["group", "metric"]):
    ai = g.set_index("seed")["AI"].reindex(thr.index)
    for col in [c for c in thr.columns if c.endswith("_pp")]:
        rho, p = stats.spearmanr(ai, thr[col])
        rows.append({"group": group, "metric": metric, "threshold_diff": col, "spearman_rho": rho, "p": p})
R3_CORR = pd.DataFrame(rows)
print("seedごとの閾値の左右差(パーセントポイント):")
print(thr.round(2).to_string())
print("\n減衰なしの非対称指数(0.5 m/s)と、閾値の左右差との順位相関(8 seed):")
print(R3_CORR.round(3).to_string(index=False))
R3_CORR.to_csv(OUT_DIR / "R3_asymmetry_vs_threshold_v2.csv", index=False)
thr.to_csv(OUT_DIR / "R3_per_seed_thresholds.csv")

## §R4. 保存

In [ ]:
with open(OUT_DIR / "R3_fall_criterion.json", "w") as f:
    json.dump({**R1_INFO, "torque_source": TAU_SOURCE}, f, indent=2, ensure_ascii=False, default=str)
import shutil
dst = "/content/drive/MyDrive/g1_cross_seed_logs/revision_outputs_R3"
os.makedirs(dst, exist_ok=True)
for p in OUT_DIR.glob("R3_*"):
    shutil.copy(p, dst)
print("保存先:", dst)
print(sorted(os.listdir(dst)))
print("\n→ revision_outputs_R3 フォルダの中身(R3_ で始まるファイル)を、まとめてアップロードしてください。")